<a href="https://colab.research.google.com/github/sahil-singh-8651/Anshu/blob/main/Data_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tweepy
import pandas as pd
import re
from textblob import TextBlob

# Replace with your API keys
bearer_token = 'AAAAAAAAAAAAAAAAAAAAAIWDxQEAAAAAa1UVgbzhyO2HDYPrsYqxW7WhwEw%3DT263er1K8F9zGbTmXz2pJzOlC9CfMWVQlrOExBbRcGyfsTYHGR'

# Authenticate using Tweepy Client
client = tweepy.Client(bearer_token=bearer_token)

# Function to scrape tweets using Twitter API v2
def scrape_tweets_v2(query, max_results=100):
    print(f"Scraping tweets about '{query}'...")
    tweets = []
    response = client.search_recent_tweets(query=query, max_results=max_results, tweet_fields=['text', 'lang'])
    if response.data:
        for tweet in response.data:
            if tweet.lang == "en":  # Filter only English tweets
                tweets.append(tweet.text)
    return tweets

# Example: Scrape tweets about Tesla stock
tweets = scrape_tweets_v2("Tesla stock", 100)
print(f"Fetched {len(tweets)} tweets.")


Scraping tweets about 'Tesla stock'...
Fetched 97 tweets.


In [ ]:
import tweepy
import pandas as pd
import re
from textblob import TextBlob
import time

# Replace with your bearer token
bearer_token = 'AAAAAAAAAAAAAAAAAAAAAIWDxQEAAAAAa1UVgbzhyO2HDYPrsYqxW7WhwEw%3DT263er1K8F9zGbTmXz2pJzOlC9CfMWVQlrOExBbRcGyfsTYHGR'

# Authenticate using Tweepy Client
client = tweepy.Client(bearer_token=bearer_token)

# Function to scrape tweets using Twitter API v2
def scrape_tweets_v2(query, max_results=100):
    print(f"Scraping tweets about '{query}'...")
    tweets = []
    while len(tweets) < max_results:
        try:
            # Fetch tweets from Twitter API
            response = client.search_recent_tweets(query=query, max_results=max_results, tweet_fields=['text', 'lang'])

            # Check if response contains data
            if response.data:
                for tweet in response.data:
                    if tweet.lang == "en":  # Filter only English tweets
                        tweets.append(tweet.text)

            # Break out of the loop if enough tweets are fetched
            break

        except tweepy.TooManyRequests:
            print("Rate limit exceeded. Waiting for 15 minutes...")
            time.sleep(15 * 60)  # Wait for 15 minutes if rate limit is exceeded

    return tweets

# Example: Scrape tweets about Tesla stock
tweets = scrape_tweets_v2("Tesla stock", 100)

# Display all fetched tweets
print(f"Fetched {len(tweets)} tweets.\n")
for tweet in tweets:
    print(tweet)

# Step 1: Preprocess and Clean Data
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)    # Remove mentions
    text = re.sub(r'#', '', text)       # Remove hashtags
    text = re.sub(r'\W', ' ', text)     # Remove non-alphanumeric characters
    return text.lower()

cleaned_tweets = [clean_text(tweet) for tweet in tweets]

# Step 2: Sentiment Analysis
def get_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        return 1  # Positive
    elif analysis.sentiment.polarity < 0:
        return -1  # Negative
    else:
        return 0  # Neutral

sentiments = [get_sentiment(tweet) for tweet in cleaned_tweets]

# Create a DataFrame for the scraped and processed data
df = pd.DataFrame({'Tweet': cleaned_tweets, 'Sentiment': sentiments})
print("\nSample Data:\n", df.head())

# Step 3: Convert Text to Numerical Features
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['Tweet']).toarray()
y = df['Sentiment']

# Step 4: Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 5: Train Machine Learning Model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 6: Evaluate the Model
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

y_pred = model.predict(X_test)
print("\nModel Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Step 7: Test Predictions on Sample Tweets
sample_tweets = ["Tesla's stock is going to the moon!",
                 "I'm bearish on Tesla right now.",
                 "Neutral about Tesla's future."]

cleaned_samples = [clean_text(tweet) for tweet in sample_tweets]
sample_features = vectorizer.transform(cleaned_samples).toarray()
predictions = model.predict(sample_features)

for tweet, sentiment in zip(sample_tweets, predictions):
    print(f"Tweet: {tweet}\nPredicted Sentiment: {'Positive' if sentiment == 1 else 'Negative' if sentiment == -1 else 'Neutral'}\n")


Scraping tweets about 'Tesla stock'...
Rate limit exceeded. Waiting for 15 minutes...
